# 🫁 Pneumonia Detection - Google Colab Master Execution Notebook

Welcome to the master execution notebook! This notebook is designed to run the entire Pneumonia Detection pipeline **exclusively on Google Colab's GPU runtime** while maintaining file persistence via **Google Drive**.

### ⚡ Architecture Overview
- **Google Drive**: Persists your source code, custom training modules (`src/`), model checkpoints (`models/`), and classification results.
- **Local Scratch Disk (`/content/data/`)**: Stores the extracted image dataset. Reading thousands of images directly from Google Drive during training is extremely slow due to Google Drive network mount limits (`FUSE`). Downloading and extracting the dataset locally on Colab's disk speeds up training **up to 10x-50x**!

---

### 🚀 Before You Start:
1. **Enable GPU**: Click on **Runtime** -> **Change runtime type** -> select **GPU** (T4, L4, or A100) -> click **Save**.
2. **Upload Folder**: Ensure this project folder (`pneumonia-detection-cnn-xray`) is uploaded directly under your Google Drive's **My Drive/** folder. 
   *   ⚠️ **CRITICAL: Exclude the `data/` and `venv/` directories when uploading!** They are not needed on Drive, saving you 1.2 GB+ of Drive space and hours of upload time. Colab handles the dataset download and extraction directly on its local scratch disk.

## Step 1: Mount Google Drive
Run this cell to authorize Google Colab to access your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Set Workspace Directory
Navigate to the project folder on Google Drive and verify the directory contents.

In [ ]:
import os

# Set project directory on Google Drive
project_dir = '/content/drive/MyDrive/pneumonia-detection-cnn-xray'

if not os.path.exists(project_dir):
    print(f"❌ ERROR: The directory '{project_dir}' was not found in your Google Drive.")
    print("Please verify that you have uploaded the 'pneumonia-detection-cnn-xray' folder directly under 'My Drive'.")
else:
    %cd {project_dir}
    print("\n📁 Workspace Directory Contents:")
    !ls -la

## Step 3: Hardware Verification & Package Installation
Let's verify that the GPU is enabled and install the project dependencies.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (GPU) Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected. Please enable GPU in Runtime -> Change runtime type.")

In [ ]:
# Install python dependencies from requirements.txt
import sys, subprocess

def pip_install(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)

try:
    import google.colab
    print("📍 Google Colab detected\nInstalling Streamlit & GitPython...\n")
    pip_install(["streamlit", "GitPython"])
    print("✅ Done (no restart needed).")
except ImportError:
    print("Local environment detected\nInstalling from requirements.txt...\n")
    pip_install(["-r", "requirements.txt"])
    print("✅ Done.")

### Step 3b: Verify Model Architecture Compilation
Run this cell to build the PyTorch models with dummy inputs and execute a mock forward pass. This ensures all model architectures compile successfully on Colab's GPU.

In [ ]:
!python test_model.py

## Step 4: High-Performance Data Extraction
This step downloads the dataset and extracts it directly onto Colab's fast local scratch disk (`/content/data`). This completely bypasses Google Drive network latency bottlenecking.

In [ ]:
# Run the download and extraction script
!python download_data.py

In [ ]:
# Verify that data files were downloaded locally
print("Local dataset structure:")
!ls -la /content/data/chest_xray
!ls -la /content/data/chest_xray/train

### Step 4b: Verify Data Loading & Dataset Pipeline
Run this cell to load one batch from your local dataset and check the image and label shapes. This ensures your data loaders and transforms are fully functional.

In [ ]:
!python test_data.py

## Step 5: Train Models (GPU Accelerated)
You can train the baseline CNN, Transfer Learning (ResNet50), or the state-of-the-art Ensemble Model (CheX-DS) on Colab's GPU.

The trained model weights are saved automatically to your Google Drive under `/content/drive/MyDrive/pneumonia-detection-cnn-xray/models/pneumonia_model.pth` so they are **safely persisted**.

### Option A: Train a Single Model
Choose which model you want to train by running one of the cells below.

In [ ]:
# Train baseline SimpleCNN model
!python main.py --model cnn

In [ ]:
# Train Transfer Learning (ResNet50) model
!python main.py --model resnet

In [ ]:
# Train CheX-DS (DenseNet + Swin Transformer Ensemble - Best Performance)
!python main.py --model chexds

### Option B: Run Full Benchmarks (All Models)
This script sequentially runs training for all models (`cnn`, `resnet`, and `chexds`) on the GPU, measuring benchmarks. The metrics summary will be logged under `/results/experiments_summary.json`.

In [ ]:
!python run_experiments.py

## Step 6: Evaluation & Result Visualization
Run this script to load the trained **CheX-DS** ensemble weights directly from Google Drive, perform inference on the test set, print the classification report, and display the Confusion Matrix & ROC Curve.

In [ ]:
%matplotlib inline
!python visualize_results.py

## Step 7: Run Streamlit Web Application inside Colab (Optional)
If you want to run the Streamlit app directly from Colab, you can expose the app port to the public web using `localtunnel`.

In [ ]:
# 1. Install localtunnel
!npm install -g localtunnel

In [ ]:
# 2. Find your Colab public IP address (you will need this password when entering the localtunnel URL)
import urllib
print("Your Colab Endpoint Password:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

In [ ]:
# 3. Run Streamlit with localtunnel
# Click the link generated by the cell below, enter the 'Endpoint Password' shown above, and submit.
!streamlit run app.py & npx localtunnel --port 8501

## Step 8: Run Streamlit App Locally (Recommended)
Since the Colab GPU has successfully finished training, the model weights `models/pneumonia_model.pth` have been written to your Google Drive.

Because single-image inference is extremely fast and does not require a GPU, you can run the Streamlit web app locally on your CPU:

1. Ensure the `pneumonia_model.pth` is synchronized/downloaded to your local `models/` directory.
2. Open a local terminal in your project directory.
3. Start the application:
   ```bash
   streamlit run app.py
   ```
4. Open `http://localhost:8501` in your browser.